In [ ]:
# PAWN warm-session POC kernel (W.0) — CPU echo, no GPU, no model.
# Proves a batch-pushed Kaggle kernel can run a long-lived internet loop and
# rendezvous with PAWN through the self-hosted PostgREST instance. PAWN
# replaces __PAWN_PAYLOAD_B64__ with base64(json) before each push. Anonymous
# requests to PostgREST get the restricted `pawn_anon` Postgres role (RLS
# limits it to the two dedicated tables) — no bearer/API key needed.
import base64, json, time, datetime
import requests  # preinstalled on Kaggle

payload = json.loads(base64.b64decode("__PAWN_PAYLOAD_B64__").decode())

SESSION_ID = payload["session_id"]
POSTGREST_URL = payload["postgrest_url"].rstrip("/")
POLL = int(payload.get("poll_interval", 3))
MAX_IMAGES = payload.get("max_images")

REST = POSTGREST_URL
HEADERS = {
    "Content-Type": "application/json",
}

def now_iso():
    return datetime.datetime.now(datetime.timezone.utc).isoformat()

def now_utc():
    return datetime.datetime.now(datetime.timezone.utc)

def parse_ts(s):
    if not s:
        return None
    try:
        return datetime.datetime.fromisoformat(s.replace("Z", "+00:00"))
    except ValueError:
        return None

def get_session():
    r = requests.get(f"{REST}/image_sessions", headers=HEADERS,
                     params={"id": f"eq.{SESSION_ID}", "select": "*"}, timeout=20)
    r.raise_for_status()
    data = r.json()
    return data[0] if data else None

def patch_session(fields):
    requests.patch(f"{REST}/image_sessions", headers=HEADERS,
                   params={"id": f"eq.{SESSION_ID}"}, json=fields, timeout=20)

def next_job():
    r = requests.get(f"{REST}/image_jobs", headers=HEADERS,
                     params={"session_id": f"eq.{SESSION_ID}", "status": "eq.queued",
                             "order": "created_at.asc", "limit": "1", "select": "*"},
                     timeout=20)
    r.raise_for_status()
    data = r.json()
    return data[0] if data else None

def patch_job(job_id, fields):
    requests.patch(f"{REST}/image_jobs", headers=HEADERS,
                   params={"id": f"eq.{job_id}"}, json=fields, timeout=20)


In [ ]:
# 'Load' step (no model in the POC) -> announce readiness + first heartbeat.
print("PAWN session POC ready:", SESSION_ID)
patch_session({"status": "ready", "heartbeat_at": now_iso()})


In [ ]:
# Work loop: heartbeat, honor stop/timer/cap, echo any pending job back.
images_done = 0
while True:
    sess = get_session()
    if sess is None:
        print("session row gone; exiting")
        break
    status = sess.get("status")
    expires = parse_ts(sess.get("expires_at"))
    cap = sess.get("max_images")
    if status in ("stopping", "ended"):
        patch_session({"status": "ended"})
        print("stop requested; exiting")
        break
    if expires is not None and now_utc() >= expires:
        patch_session({"status": "ended"})
        print("timer expired; exiting")
        break
    if cap is not None and images_done >= cap:
        patch_session({"status": "ended"})
        print("image cap reached; exiting")
        break
    patch_session({"heartbeat_at": now_iso()})
    job = next_job()
    if not job:
        time.sleep(POLL)
        continue
    job_id = job["id"]
    patch_job(job_id, {"status": "running", "started_at": now_iso()})
    try:
        # CPU echo: no model. Echo the prompt back as the 'image' payload so the
        # full rendezvous (queue -> pick up -> write result -> read back) is proven.
        echoed = base64.b64encode(("ECHO: " + (job.get("prompt") or "")).encode()).decode()
        patch_job(job_id, {"status": "done", "image_b64": echoed,
                           "mime": "text/plain", "via": "kaggle:session-poc",
                           "done_at": now_iso()})
        images_done += 1
        patch_session({"images_done": images_done})
    except Exception as e:
        patch_job(job_id, {"status": "error", "error": str(e), "done_at": now_iso()})
